In [ ]:
# Write your code here
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
def find_imagefolder_root(base_path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
    for root, dirs, files in os.walk(base_path):
        if len(dirs) >= 2:
            has_img = any(os.path.splitext(f)[1].lower() in exts for f in files)
            if has_img:
                return root
    return base_path

root_dir = find_imagefolder_root(path)

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

full_dataset_for_split = ImageFolder(root=root_dir, transform=transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
]))

num_samples = len(full_dataset_for_split)
indices = torch.randperm(num_samples).tolist()
split = int(0.8 * num_samples)
train_idx, test_idx = indices[:split], indices[split:]

train_dataset = Subset(ImageFolder(root=root_dir, transform=train_transform), train_idx)
test_dataset  = Subset(ImageFolder(root=root_dir, transform=test_transform),  test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_names = full_dataset_for_split.classes
print("Classes:", class_names)
print(f"Training samples: {len(train_dataset)}, Testing samples: {len(test_dataset)}")



In [ ]:
# Display some sample images with their labels
data_iter = iter(train_loader)
images, labels = next(data_iter)

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()])
    ax.axis("off")
plt.show()

print("Shape of one image tensor:", images[0].shape)

In [ ]:
# Write your code here
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")
print("Path to dataset files:", path)

# Part 1: Load and Prepare Data (4 points)

def find_imagefolder_root(base_path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
    for root, dirs, files in os.walk(base_path):
        if len(dirs) >= 2:
            has_img = any(os.path.splitext(f)[1].lower() in exts for f in files)
            if has_img:
                return root
    return base_path

root_dir = find_imagefolder_root(path)

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

full_dataset_for_split = ImageFolder(root=root_dir, transform=transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
]))

num_samples = len(full_dataset_for_split)
indices = torch.randperm(num_samples).tolist()
split = int(0.8 * num_samples)
train_idx, test_idx = indices[:split], indices[split:]

train_dataset = Subset(ImageFolder(root=root_dir, transform=train_transform), train_idx)
test_dataset  = Subset(ImageFolder(root=root_dir, transform=test_transform),  test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_names = full_dataset_for_split.classes
print("Classes:", class_names)
print(f"Training samples: {len(train_dataset)}, Testing samples: {len(test_dataset)}")

# Display some sample images with their labels
data_iter = iter(train_loader)
images, labels = next(data_iter)

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()])
    ax.axis("off")
plt.show()

print("Shape of one image tensor:", images[0].shape)

# Part 2: Build the CNN Model (3 points)

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)  # 16x16x16
        x = self.conv2(x)  # 32x8x8
        x = self.conv3(x)  # 64x4x4
        x = self.conv4(x)  # 128x2x2
        x = self.conv5(x)  # 256x2x2
        x = self.classifier(x)
        return x

# Part 3: Training and Validation Functions

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in tqdm(dataloader):
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    return total_loss / len(dataloader), 100.0 * correct / total

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in dataloader:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        outputs = model(imgs)
        loss = criterion(outputs, lbls)

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    return total_loss / len(dataloader), 100.0 * correct / total

# Part 4: Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 2

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.2f}% | Val Loss={va_loss:.4f}, Val Acc={va_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accs, marker='o', label="Train Acc")
plt.plot(range(1, num_epochs+1), val_accs, marker='o', label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy")
plt.legend()
plt.show()

# Part 5: Bonus - Residual Connection

class PotatoCNNResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNNResidual, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )

        self.conv4_pre = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.conv4_pool = nn.MaxPool2d(2, 2)

        self.skip_pool = nn.MaxPool2d(2, 2)          # 8x8 -> 4x4
        self.skip_proj = nn.Conv2d(32, 128, 1)       # 32ch -> 128ch

        self.conv5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)          # 16x16x16
        x2 = self.conv2(x)         # 32x8x8

        x3 = self.conv3(x2)        # 64x4x4

        x4 = self.conv4_pre(x3)    # 128x4x4

        skip = self.skip_proj(self.skip_pool(x2))  # 128x4x4
        x4 = x4 + skip

        x4 = self.conv4_pool(x4)   # 128x2x2
        x5 = self.conv5(x4)        # 256x2x2
        out = self.classifier(x5)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = PotatoCNNResidual(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_res.parameters(), lr=0.001)
num_epochs = 10

train_losses_r, val_losses_r = [], []
train_accs_r, val_accs_r = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model_res, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model_res, test_loader, criterion, device)

    train_losses_r.append(tr_loss)
    val_losses_r.append(va_loss)
    train_accs_r.append(tr_acc)
    val_accs_r.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.2f}% | Val Loss={va_loss:.4f}, Val Acc={va_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses_r, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses_r, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accs_r, marker='o', label="Train Acc")
plt.plot(range(1, num_epochs+1), val_accs_r, marker='o', label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Residual Accuracy")
plt.legend()
plt.show()



In [ ]:
# Write your code here
class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)  # 16x16x16
        x = self.conv2(x)  # 32x8x8
        x = self.conv3(x)  # 64x4x4
        x = self.conv4(x)  # 128x2x2
        x = self.conv5(x)  # 256x2x2
        x = self.classifier(x)
        return x


In [ ]:
# Write your code here
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in tqdm(dataloader):
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    return total_loss / len(dataloader), 100.0 * correct / total

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in dataloader:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        outputs = model(imgs)
        loss = criterion(outputs, lbls)

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    return total_loss / len(dataloader), 100.0 * correct / total


In [ ]:
# Write your code here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.2f}% | Val Loss={va_loss:.4f}, Val Acc={va_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accs, marker='o', label="Train Acc")
plt.plot(range(1, num_epochs+1), val_accs, marker='o', label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy")
plt.legend()
plt.show()


In [ ]:
# Write your code here
class PotatoCNNResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNNResidual, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )

        self.conv4_pre = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.conv4_pool = nn.MaxPool2d(2, 2)

        self.skip_pool = nn.MaxPool2d(2, 2)          # 8x8 -> 4x4
        self.skip_proj = nn.Conv2d(32, 128, 1)       # 32ch -> 128ch

        self.conv5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)          # 16x16x16
        x2 = self.conv2(x)         # 32x8x8

        x3 = self.conv3(x2)        # 64x4x4

        x4 = self.conv4_pre(x3)    # 128x4x4

        skip = self.skip_proj(self.skip_pool(x2))  # 128x4x4
        x4 = x4 + skip

        x4 = self.conv4_pool(x4)   # 128x2x2
        x5 = self.conv5(x4)        # 256x2x2
        out = self.classifier(x5)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = PotatoCNNResidual(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_res.parameters(), lr=0.001)
num_epochs = 10

train_losses_r, val_losses_r = [], []
train_accs_r, val_accs_r = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model_res, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model_res, test_loader, criterion, device)

    train_losses_r.append(tr_loss)
    val_losses_r.append(va_loss)
    train_accs_r.append(tr_acc)
    val_accs_r.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.2f}% | Val Loss={va_loss:.4f}, Val Acc={va_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses_r, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses_r, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accs_r, marker='o', label="Train Acc")
plt.plot(range(1, num_epochs+1), val_accs_r, marker='o', label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Residual Accuracy")
plt.legend()
plt.show()
